In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_telco_silver")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-61cb428b-fbed-4ce0-abe4-3fc358d07a78;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (737ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (73ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (180ms)
:: resolution report :: resolve 2133ms :: artifacts dl 9

In [3]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [4]:
path = "s3a://bronze/base_telco/"
df_base_telco = spark.read.parquet(path)
df_base_telco.show(5, truncate=False)

25/12/28 13:22:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/28 13:22:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:=============================>                             (1 + 1) / 2]

+-----------+------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72|var_73|var_74|var_75|var_76|var_77|var_78|var_79|var_80|var_81|var_82|var_83|var_84|

In [5]:
df_base_telco.createOrReplaceTempView("raw_00")

In [6]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

[Stage 6:=======================================>                   (2 + 1) / 3]

+------+------------+-------------+
|SAFRA |total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|219860      |219860       |
|202411|240520      |240520       |
|202412|241453      |241453       |
|202501|233710      |233710       |
|202502|214665      |214665       |
+------+------------+-------------+
only showing top 5 rows



In [7]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(NUM_CPF as) as NUM_CPF,
cast(SAFRA as) as SAFRA,
cast(FLAG_INSTALACAO as) as FLAG_INSTALACAO,
cast(FPD as) as FPD,
cast(PROD as) as PROD,
cast(flag_mig2 as) as flag_mig2,
cast(var_26 as) as var_26,
cast(var_27 as) as var_27,
cast(var_28 as) as var_28,
cast(var_29 as) as var_29,
cast(var_30 as) as var_30,
cast(var_31 as) as var_31,
cast(var_32 as) as var_32,
cast(var_33 as) as var_33,
cast(var_34 as) as var_34,
cast(var_35 as) as var_35,
cast(var_36 as) as var_36,
cast(var_37 as) as var_37,
cast(var_38 as) as var_38,
cast(var_39 as) as var_39,
cast(var_40 as) as var_40,
cast(var_41 as) as var_41,
cast(var_42 as) as var_42,
cast(var_43 as) as var_43,
cast(var_44 as) as var_44,
cast(var_45 as) as var_45,
cast(var_46 as) as var_46,
cast(var_47 as) as var_47,
cast(var_48 as) as var_48,
cast(var_49 as) as var_49,
cast(var_50 as) as var_50,
cast(var_51 as) as var_51,
cast(var_52 as) as var_52,
cast(var_53 as) as var_53,
cast(var_54 as) as var_54,
cast(var_55 a

In [8]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            cast(NUM_CPF as string) as NUM_CPF,
            cast(SAFRA as int) as SAFRA,
            try_cast(FLAG_INSTALACAO as int) as FLAG_INSTALACAO,
            try_cast(FPD as int) as FPD,
            cast(PROD as string) as PROD,
            cast(flag_mig2 as string) as flag_mig2,
            try_cast(var_26 as int) as var_26,
            try_cast(var_27 as int) as var_27,
            try_cast(var_28 as decimal(10,2)) as var_28,
            try_cast(var_29 as decimal(10,2)) as var_29,
            try_cast(var_30 as decimal(10,2)) as var_30,
            try_cast(var_31 as decimal(10,2)) as var_31,
            try_cast(var_32 as decimal(10,2)) as var_32,
            try_cast(var_33 as decimal(10,2)) as var_33,
            try_cast(var_34 as decimal(10,2)) as var_34,
            try_cast(var_35 as decimal(10,2)) as var_35,
            try_cast(var_36 as decimal(10,2)) as var_36,
            try_cast(var_37 as decimal(10,2)) as var_37,
            try_cast(var_38 as decimal(10,2)) as var_38,
            try_cast(var_39 as decimal(10,2)) as var_39,
            try_cast(var_40 as decimal(10,2)) as var_40,
            try_cast(var_41 as decimal(10,2)) as var_41,
            try_cast(var_42 as decimal(10,2)) as var_42,
            try_cast(var_43 as decimal(10,2)) as var_43,
            try_cast(var_44 as decimal(10,2)) as var_44,
            try_cast(var_45 as decimal(10,2)) as var_45,
            try_cast(var_46 as decimal(10,2)) as var_46,
            try_cast(var_47 as decimal(10,2)) as var_47,
            try_cast(var_48 as decimal(10,2)) as var_48,
            try_cast(var_49 as decimal(10,2)) as var_49,
            try_cast(var_50 as decimal(10,2)) as var_50,
            try_cast(var_51 as decimal(10,2)) as var_51,
            try_cast(var_52 as decimal(10,2)) as var_52,
            try_cast(var_53 as decimal(10,2)) as var_53,
            try_cast(var_54 as decimal(10,2)) as var_54,
            try_cast(var_55 as decimal(10,2)) as var_55,
            try_cast(var_56 as decimal(10,2)) as var_56,
            try_cast(var_57 as decimal(10,2)) as var_57,
            try_cast(var_58 as decimal(10,2)) as var_58,
            try_cast(var_59 as decimal(10,2)) as var_59,
            try_cast(var_60 as decimal(10,2)) as var_60,
            try_cast(var_61 as decimal(10,2)) as var_61,
            try_cast(var_62 as decimal(10,2)) as var_62,
            try_cast(var_63 as decimal(10,2)) as var_63,
            try_cast(var_64 as decimal(10,2)) as var_64,
            try_cast(var_65 as decimal(10,2)) as var_65,
            try_cast(var_66 as decimal(10,2)) as var_66,
            try_cast(var_67 as decimal(10,2)) as var_67,
            try_cast(var_68 as decimal(10,2)) as var_68,
            try_cast(var_69 as decimal(10,2)) as var_69,
            try_cast(var_70 as decimal(10,2)) as var_70,
            try_cast(var_71 as decimal(10,2)) as var_71,
            try_cast(var_72 as decimal(10,2)) as var_72,
            try_cast(var_73 as decimal(10,2)) as var_73,
            try_cast(var_74 as int) as var_74,
            try_cast(var_75 as int) as var_75,
            try_cast(var_76 as int) as var_76,
            try_cast(var_77 as int) as var_77,
            try_cast(var_78 as int) as var_78,
            try_cast(var_79 as decimal(10,2)) as var_79,
            try_cast(var_80 as decimal(10,2)) as var_80,
            try_cast(var_81 as decimal(10,2)) as var_81,
            try_cast(var_82 as int) as var_82,
            try_cast(var_83 as int) as var_83,
            try_cast(var_84 as int) as var_84,
            try_cast(var_85 as int) as var_85,
            try_cast(var_86 as decimal(10,2)) as var_86,
            try_cast(var_87 as int) as var_87,
            try_cast(var_88 as int) as var_88,
            try_cast(var_89 as int) as var_89,
            try_cast(var_90 as int) as var_90,
            try_cast(var_91 as int) as var_91,
            try_cast(var_92 as int) as var_92,
            try_cast(var_93 as int) as var_93,
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

1367104

In [9]:
lake.show(5)

+-----------+------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+--------------+
|    NUM_CPF| SAFRA|FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72| var_73|var_74|var_75|var_76|var_77|var_78|var_79|var_80|var_81|var_

In [10]:
for col in lake.columns:
    agg_result = lake.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = lake.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

FPD: 45936 nulos (3.36%)
flag_mig2: 58130 nulos (4.25%)
var_26: 1295 nulos (0.09%)
var_27: 1295 nulos (0.09%)


var_28: 1295 nulos (0.09%)
var_29: 1295 nulos (0.09%)
var_30: 1295 nulos (0.09%)
var_31: 1295 nulos (0.09%)
var_32: 1295 nulos (0.09%)
var_33: 1295 nulos (0.09%)
var_34: 1295 nulos (0.09%)
var_35: 1295 nulos (0.09%)
var_36: 1295 nulos (0.09%)
var_37: 1295 nulos (0.09%)
var_38: 1295 nulos (0.09%)
var_39: 1295 nulos (0.09%)
var_40: 1295 nulos (0.09%)
var_41: 1295 nulos (0.09%)
var_42: 1295 nulos (0.09%)
var_43: 1295 nulos (0.09%)
var_44: 1295 nulos (0.09%)
var_45: 1295 nulos (0.09%)
var_46: 1295 nulos (0.09%)
var_47: 1295 nulos (0.09%)
var_48: 1295 nulos (0.09%)
var_49: 1295 nulos (0.09%)
var_50: 1295 nulos (0.09%)
var_51: 1295 nulos (0.09%)
var_52: 1295 nulos (0.09%)
var_53: 1295 nulos (0.09%)
var_54: 1295 nulos (0.09%)
var_55: 1295 nulos (0.09%)
var_56: 1295 nulos (0.09%)
var_57: 1295 nulos (0.09%)
var_58: 1295 nulos (0.09%)
var_59: 1295 nulos (0.09%)
var_60: 1295 nulos (0.09%)
var_61: 1295 nulos (0.09%)
var_62: 1295 nulos (0.09%)
var_63: 1295 nulos (0.09%)
var_64: 1295 nulos (0.09%)
v

var_82: 154787 nulos (11.32%)


var_90: 379644 nulos (27.77%)


In [11]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5)

[Stage 467:======================================>                  (2 + 1) / 3]

+------+------------+-------------+
| SAFRA|total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|      219860|       219860|
|202411|      240520|       240520|
|202412|      241453|       241453|
|202501|      233710|       233710|
|202502|      214665|       214665|
+------+------------+-------------+
only showing top 5 rows



In [12]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_telco/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [13]:
spark.stop()